# LeetCode #181 — Employees Earning More Than Their Managers

**Vị trí 41 trong danh sách Easy — SQL nhập môn từ nền tảng.**  
Đề gốc: https://leetcode.com/problems/employees-earning-more-than-their-managers/

**Điều đặc biệt:** Đây là bài **Database**; LeetCode chấm **câu lệnh SQL**, không chấm class Python. Notebook dùng **Python 3 + SQLite (`sqlite3`)** chỉ để dựng bảng, chạy SQL thật, mô phỏng và tự kiểm thử. Không cần cài MySQL hay pandas.

**Mục tiêu sau bài:**
1. Đọc bảng nhân viên, hiểu `id`, `managerId`, khóa chính, khóa tham chiếu và `NULL`.
2. Dùng `SELECT`, `FROM`, `WHERE`, `JOIN`, bí danh `e`/`m` và `AS` để đặt tên cột.
3. Hiểu **self join (tự ghép bảng)**: cùng bảng Employee đóng hai vai trò nhân viên và quản lý.
4. Tự trace từng cặp nhân viên–quản lý, đối chiếu lương và tìm nhân viên lương **cao hơn nghiêm ngặt**.
5. Biết vì sao `INNER JOIN` hợp lý; phát hiện lỗi so lương sai người, sai toán tử, nhầm tên cột.
6. Giải biến thể, luyện 6 bài và tự viết đúng câu SQL nộp LeetCode.

**Đề nghị học:** đoán output trước mỗi cell. Phần “TODO” không được chạy tự động; đáp án đầy đủ đặt cuối notebook.

## 1. Đề bài muốn tìm ai?

Mỗi nhân viên có **mã định danh**, **tên**, **lương** và **mã người quản lý trực tiếp**. Những người quản lý cũng là nhân viên trong cùng bảng.

| id | name | salary | managerId |
|---:|---|---:|---:|
| 1 | Joe | 70000 | 3 |
| 2 | Henry | 80000 | 4 |
| 3 | Sam | 60000 | NULL |
| 4 | Max | 90000 | NULL |

- Joe (`id=1`) có `managerId=3` → quản lý là Sam (lương 60000). `70000 > 60000` → **chọn Joe**.
- Henry (`id=2`) có `managerId=4` → quản lý là Max (lương 90000). `80000 > 90000` sai → bỏ Henry.
- Sam và Max không có người quản lý trong dữ liệu mẫu (`managerId=NULL`) → không thuộc kết quả.

**Output bắt buộc:** đúng **một cột mang tên `Employee`**, gồm dòng `Joe`. Thứ tự dòng bất kỳ.

## 2. Bốn từ vựng SQL cơ bản: table, row, column, primary key

Một **table (bảng)** chứa nhiều **row (dòng)**, mỗi dòng có các **column (cột)** theo schema.

- `id` là **primary key (khóa chính)**: mỗi nhân viên có một ID riêng, không trùng.
- `name` là tên; **không được tự giả định** rằng hai người không thể trùng tên.
- `salary` là mức lương dạng số nguyên; trong SQL dùng so sánh `>`.
- `managerId` là ID người quản lý trong cùng bảng Employee. Nó **không phải** tên, không phải lương và không phải ID của phòng ban.

**Quan hệ cốt lõi:** `employee.managerId = manager.id`. Từ đó ta biết lấy mức lương của đúng người quản lý nào.

## 3. Python `sqlite3`: ta sẽ chạy SQL thật trong notebook

Bản thân Jupyter notebook thường chạy **Python kernel**, không hiểu trực tiếp các dòng `SELECT ... FROM ...` viết trần trong code cell. Vì vậy ta truyền **chuỗi chứa SQL** vào kết nối SQLite:

```python
conn.execute('SELECT ... FROM Employee')
```

- `sqlite3.connect(':memory:')`: tạo database tạm trong RAM.
- `CREATE TABLE`: dựng cấu trúc bảng.
- `INSERT`: thêm dữ liệu.
- `SELECT`: lấy kết quả.

**Chú ý:** `CREATE TABLE` và `INSERT` chỉ là bước chuẩn bị để học offline; khi nộp LeetCode, hệ thống đã có bảng Employee nên **chỉ nộp SELECT**.

In [1]:
import sqlite3
conn181=sqlite3.connect(':memory:')
print('Kết nối SQLite đã sẵn sàng:',sqlite3.sqlite_version)

Kết nối SQLite đã sẵn sàng: 3.46.1


## 4. Dựng đúng bảng Employee theo schema bài gốc

Đề mô tả các cột SQL:

```text
Employee(id int, name varchar, salary int, managerId int)
```

Để thực hành trong SQLite, ta viết `INTEGER` và `TEXT` thay cho `int` và `varchar` — đủ để biểu diễn dữ liệu của bài. `managerId` không dùng `PRIMARY KEY` vì **nhiều nhân viên có thể cùng một quản lý**.

Một quản lý có thể cũng là nhân viên của quản lý khác, tức bản thân quan hệ này có thể tạo một **chuỗi cấp trên**. Bài chỉ hỏi **quản lý trực tiếp** nên ta ghép đúng một lần.

In [2]:
conn181.executescript("""
CREATE TABLE Employee (
    id INTEGER PRIMARY KEY,
    name TEXT,
    salary INTEGER,
    managerId INTEGER
);
""")
print('Đã tạo bảng Employee(id, name, salary, managerId)')

Đã tạo bảng Employee(id, name, salary, managerId)


## 5. Nạp 4 dòng của ví dụ LeetCode

`INSERT INTO Employee VALUES (...)` đưa một dòng nhân viên vào bảng. Với Python, `executemany` cho phép chèn nhiều dòng một cách gọn và an toàn bằng dấu `?` tương ứng các giá trị.

Ở Python, `None` khi truyền vào SQLite trở thành SQL `NULL`. Đây là ví dụ ý nghĩa thực tế của việc giá trị thiếu: Sam và Max **không có managerId**. Không được hiểu NULL là số 0.

In [3]:
employee_example=[
    (1,'Joe',70000,3),
    (2,'Henry',80000,4),
    (3,'Sam',60000,None),
    (4,'Max',90000,None),
]
conn181.executemany('INSERT INTO Employee VALUES (?,?,?,?)',employee_example)
conn181.commit()
print('Đã chèn',len(employee_example),'nhân viên')

Đã chèn 4 nhân viên


## 6. Hàm giúp nhìn kết quả SQL (Python chỉ để hiển thị)

Ta cần một cách **in rõ** tên cột và dòng trả về. `show_sql_181` nhận chuỗi câu SQL, gửi cho SQLite và in các dòng.

`cursor.description` là mô tả các cột kết quả. `fetchall()` là danh sách các dòng. Dòng có SQL NULL được biểu diễn thành `None` trong Python; đó chỉ là cách Python biểu diễn dữ liệu SQLite trả về.

Với SQL chỉ truy vấn dữ liệu, ta sẽ có kết quả để quan sát và so sánh với dự đoán.

In [4]:
def show_sql_181(query,db=conn181):
    cursor=db.execute(query)
    columns=[desc[0] for desc in cursor.description]
    rows=cursor.fetchall()
    print('Cột:',columns)
    for row in rows:
        print('  ',row)
    print('Số dòng:',len(rows))
    return rows

show_sql_181('SELECT id, name, salary, managerId FROM Employee ORDER BY id;')

Cột: ['id', 'name', 'salary', 'managerId']
   (1, 'Joe', 70000, 3)
   (2, 'Henry', 80000, 4)
   (3, 'Sam', 60000, None)
   (4, 'Max', 90000, None)
Số dòng: 4


[(1, 'Joe', 70000, 3),
 (2, 'Henry', 80000, 4),
 (3, 'Sam', 60000, None),
 (4, 'Max', 90000, None)]

## 7. `SELECT` và `FROM` — hãy học trên một bảng trước

Câu:
```sql
SELECT name, salary
FROM Employee;
```
được đọc là: “Trong bảng Employee, cho tôi xem tên và lương.”

- `SELECT`: tên **các cột muốn xem**.
- `FROM`: bảng **chứa dữ liệu ban đầu**.
- `ORDER BY id`: sắp xếp để dễ nhìn trong notebook; câu nộp bài **không cần** vì đề cho phép kết quả bất kỳ thứ tự.

Không dùng `SELECT *` khi nộp vì đề chỉ muốn cột `Employee` chứa tên, không muốn id, salary hoặc managerId.

In [5]:
show_sql_181('SELECT name, salary FROM Employee ORDER BY id;')
show_sql_181('SELECT id, managerId FROM Employee ORDER BY id;')

Cột: ['name', 'salary']
   ('Joe', 70000)
   ('Henry', 80000)
   ('Sam', 60000)
   ('Max', 90000)
Số dòng: 4
Cột: ['id', 'managerId']
   (1, 3)
   (2, 4)
   (3, None)
   (4, None)
Số dòng: 4


[(1, 3), (2, 4), (3, None), (4, None)]

## 8. `WHERE`: so sánh lớn hơn bằng `>`

Trong SQL:

```sql
SELECT name FROM Employee WHERE salary > 70000;
```

lấy những nhân viên có **lương > 70000**. Dấu `>` là “lớn hơn nghiêm ngặt”, **không bao gồm bằng**. Dấu `>=` là “lớn hơn hoặc bằng”.

Nhưng chú ý: `WHERE salary > 70000` **không giải được bài gốc** vì quản lý của mỗi người có mức lương khác nhau. Ta không thể so với một **ngưỡng cố định**. Ta cần tìm **lương quản lý tương ứng từng nhân viên** rồi mới so sánh.

In [6]:
show_sql_181('SELECT name, salary FROM Employee WHERE salary > 70000 ORDER BY id;')
show_sql_181('SELECT name, salary FROM Employee WHERE salary >= 70000 ORDER BY id;')

Cột: ['name', 'salary']
   ('Henry', 80000)
   ('Max', 90000)
Số dòng: 2
Cột: ['name', 'salary']
   ('Joe', 70000)
   ('Henry', 80000)
   ('Max', 90000)
Số dòng: 3


[('Joe', 70000), ('Henry', 80000), ('Max', 90000)]

## 9. `managerId` là đường nối từ nhân viên tới quản lý

Hãy nhìn cặp:

- Joe có `id = 1`, `managerId = 3`.
- Trong **cùng bảng Employee**, Sam có `id = 3`.
- Do đó **Sam là quản lý của Joe**.

Công thức ghép:

```text
employee.managerId = manager.id
```

Đây không phải phép so sánh **lương**, mà là điều kiện **tìm đúng người**. Chỉ **sau khi** xác định được người quản lý, ta mới so sánh `employee.salary > manager.salary`.

Hai phép so sánh này có vai trò hoàn toàn khác nhau: `JOIN ON` để **ghép dòng**, `WHERE` để **lọc kết quả**.

In [7]:
for id_, name, salary, manager_id in employee_example:
    print(f'{name:<6} (id={id_}) báo cáo cho managerId={manager_id}')

Joe    (id=1) báo cáo cho managerId=3
Henry  (id=2) báo cáo cho managerId=4
Sam    (id=3) báo cáo cho managerId=None
Max    (id=4) báo cáo cho managerId=None


## 10. Cách trực giác: tìm quản lý bằng Python rồi so lương

Trước SQL self join, hãy thử suy nghĩ như một người cầm bảng tính:

1. Chọn một nhân viên đang xét.
2. Tìm dòng có `id` bằng `managerId` của nhân viên đó.
3. So hai số lương.
4. Nếu lương nhân viên lớn hơn thì lấy tên.

Đây là **baseline** giúp suy ra tự nhiên câu JOIN. Ta dùng danh sách `employee_example` và `for` lồng nhau để minh họa. Với `n` nhân viên, cách tìm tuyến tính từng quản lý có thể cần **O(n²)** thời gian; dùng dictionary có thể giảm tìm kiếm trung bình, nhưng ở bài SQL ta muốn database làm phép nối.

In [8]:
baseline_result=[]
for emp_id,emp_name,emp_salary,manager_id in employee_example:
    for mgr_id,mgr_name,mgr_salary,_ in employee_example:
        if manager_id==mgr_id and emp_salary>mgr_salary:
            baseline_result.append(emp_name)
print('Người có lương > quản lý:',baseline_result)
assert baseline_result==['Joe']

Người có lương > quản lý: ['Joe']


## 11. Bảng trace bằng tay: phân tích từng dòng

**Mô phỏng dữ liệu chính thức:**

| Nhân viên | Lương NV | `managerId` | Người quản lý tìm được | Lương QL | So sánh | Kết quả |
|---|---:|---:|---|---:|---|---|
| Joe | 70000 | 3 | Sam | 60000 | 70000 > 60000 | **Chọn Joe** |
| Henry | 80000 | 4 | Max | 90000 | 80000 > 90000 sai | Bỏ |
| Sam | 60000 | NULL | Không có | — | Không thể so | Bỏ |
| Max | 90000 | NULL | Không có | — | Không thể so | Bỏ |

Kết quả chỉ có Joe. Nếu bạn có thể dựng lại **hai dòng Joe, Henry bằng tay** thì đã hiểu lõi bài toán; SQL chỉ tự động hóa quá trình ghép và lọc.

## 12. Vì sao JOIN phải dùng cùng một bảng hai lần?

Đề chỉ cung cấp **một bảng Employee**. Thế nhưng ta cần cùng lúc hai vai trò:

- `e`: dòng **employee** (nhân viên đang được xét).
- `m`: dòng **manager** (quản lý của nhân viên đó).

Cùng bảng được đọc **hai lần với hai bí danh khác nhau**. Kỹ thuật này là **self join (tự nối một bảng với chính nó)**.

Điều kiện `ON e.managerId = m.id` nối đúng một nhân viên tới quản lý của họ. Không có bảng `Manager` riêng nào trong đề — nếu tự viết `JOIN Manager` sẽ lỗi “không có bảng”.

## 13. `AS e`, `AS m` và dấu chấm `e.name` là gì?

`FROM Employee AS e JOIN Employee AS m` nói rằng ta dùng Employee với hai **bí danh (alias)**.

- `e.name`: tên **nhân viên**.
- `m.name`: tên **quản lý**.
- `e.salary`: lương **nhân viên**.
- `m.salary`: lương **quản lý**.

Vì cả e và m đều có cột `name`, `id`, `salary`, nếu viết chỉ `name` thường sẽ gặp lỗi **ambiguous column name** (không rõ name của ai). Dấu chấm là cách ghi **bí danh bảng + tên cột**, không phải truy cập thuộc tính đối tượng Python.

In [9]:
SQL_DEBUG_JOIN="""
SELECT e.id AS emp_id, e.name AS employee_name,
       e.managerId, m.id AS manager_id, m.name AS manager_name,
       e.salary AS employee_salary, m.salary AS manager_salary
FROM Employee AS e
JOIN Employee AS m ON e.managerId = m.id
ORDER BY e.id;
"""
show_sql_181(SQL_DEBUG_JOIN)

Cột: ['emp_id', 'employee_name', 'managerId', 'manager_id', 'manager_name', 'employee_salary', 'manager_salary']
   (1, 'Joe', 3, 3, 'Sam', 70000, 60000)
   (2, 'Henry', 4, 4, 'Max', 80000, 90000)
Số dòng: 2


[(1, 'Joe', 3, 3, 'Sam', 70000, 60000),
 (2, 'Henry', 4, 4, 'Max', 80000, 90000)]

## 14. `INNER JOIN` ghép các cặp nhân viên–quản lý

Với SQL:

```sql
FROM Employee AS e
INNER JOIN Employee AS m
    ON e.managerId = m.id
```

`INNER JOIN` chỉ giữ **các cặp có quản lý tìm được**. Trong mẫu, ta thu được:

- `(Joe, Sam)`
- `(Henry, Max)`

Sam/Max không có `managerId` nên không có cặp khớp, bị loại. Điều này **đúng** vì bài hỏi “cao hơn **quản lý của mình**”, không hỏi người không có quản lý.

`JOIN` không có chữ `INNER` cũng mặc định là `INNER JOIN` trong SQL của bài này.

In [10]:
SQL_MANAGER_PAIRS="""
SELECT e.name AS employee_name, m.name AS manager_name
FROM Employee AS e
INNER JOIN Employee AS m ON e.managerId=m.id
ORDER BY e.id;
"""
show_sql_181(SQL_MANAGER_PAIRS)

Cột: ['employee_name', 'manager_name']
   ('Joe', 'Sam')
   ('Henry', 'Max')
Số dòng: 2


[('Joe', 'Sam'), ('Henry', 'Max')]

## 15. Đọc và trace `JOIN` từ trái sang phải

Khi SQL xử lý cặp `(e,m)`, điều kiện `ON e.managerId = m.id` phải đúng.

| e đang xét | `e.managerId` | m đang xét | `m.id` | Có ghép? |
|---|---:|---|---:|---|
| Joe | 3 | Henry | 2 | Không |
| Joe | 3 | Sam | 3 | **Có** |
| Henry | 4 | Sam | 3 | Không |
| Henry | 4 | Max | 4 | **Có** |

Khi hiểu điều kiện **ghép**, ta có thể tự suy ra bảng trung gian. **Chưa cần vội thêm `WHERE`**: hãy chắc chắn `JOIN` đã tìm đúng người quản lý trước.

In [11]:
show_sql_181("""SELECT e.name,m.name,e.managerId,m.id
FROM Employee e JOIN Employee m ON e.managerId=m.id ORDER BY e.id;""")

Cột: ['name', 'name', 'managerId', 'id']
   ('Joe', 'Sam', 3, 3)
   ('Henry', 'Max', 4, 4)
Số dòng: 2


[('Joe', 'Sam', 3, 3), ('Henry', 'Max', 4, 4)]

## 16. Bước đột phá: thêm `WHERE e.salary > m.salary`

Sau JOIN, mỗi dòng đã có **lương của nhân viên và quản lý**. Giờ bài toán “lương nhân viên cao hơn quản lý” viết thành:

```sql
WHERE e.salary > m.salary
```

**Tại sao dấu `>` không phải `>=`?** Vì “earn more than” nghĩa là **cao hơn**, không phải “ít nhất bằng”. Nếu hai người có lương bằng nhau thì nhân viên **không** thuộc kết quả.

Đọc phép lọc: “Trong những cặp đã ghép, giữ cặp mà lương dòng e lớn hơn lương dòng m.”

In [12]:
SQL_FILTERED_PAIRS="""
SELECT e.name AS employee_name, e.salary AS employee_salary,
       m.name AS manager_name, m.salary AS manager_salary
FROM Employee e
JOIN Employee m ON e.managerId=m.id
WHERE e.salary>m.salary;
"""
show_sql_181(SQL_FILTERED_PAIRS)

Cột: ['employee_name', 'employee_salary', 'manager_name', 'manager_salary']
   ('Joe', 70000, 'Sam', 60000)
Số dòng: 1


[('Joe', 70000, 'Sam', 60000)]

## 17. Câu hỏi cuối: chỉ cần tên nhân viên, không cần lương

Sau khi JOIN + WHERE chọn đúng người, `SELECT` chỉ xuất **một cột** mang tên `Employee`:

```sql
SELECT e.name AS Employee
```

- `e.name` là **tên người làm thuê**, không phải `m.name`.
- `AS Employee` là **bí danh cột kết quả** đúng yêu cầu đề (chữ `Employee` viết hoa E).
- Ta **không** xuất salary, managerId hoặc manager_name.

SQL có thể đọc như một đường suy luận: **ghép đúng người quản lý → so đúng lương → xuất đúng cột tên**.

In [13]:
SQL_SUBMIT_181="""
SELECT e.name AS Employee
FROM Employee AS e
JOIN Employee AS m
    ON e.managerId = m.id
WHERE e.salary > m.salary;
"""
show_sql_181(SQL_SUBMIT_181)

Cột: ['Employee']
   ('Joe',)
Số dòng: 1


[('Joe',)]

## 18. Trace từng cặp có cột quyết định `qualifies`

Ta thêm một cột tính điều kiện để **nhìn rõ True/False** (SQLite thường biểu diễn True/False bằng 1/0):

```sql
(e.salary > m.salary) AS qualifies
```

- Joe–Sam: 70000 > 60000 → **1** (đúng).
- Henry–Max: 80000 > 90000 → **0** (sai).

Cột này **chỉ để debug**, không thuộc kết quả nộp bài. Khi đã hiểu, đưa điều kiện vào WHERE để giữ những dòng `qualifies=1`.

In [14]:
SQL_TRACE_181="""
SELECT e.name AS employee, e.salary AS emp_salary,
       m.name AS manager, m.salary AS manager_salary,
       (e.salary>m.salary) AS qualifies
FROM Employee e JOIN Employee m ON e.managerId=m.id
ORDER BY e.id;
"""
show_sql_181(SQL_TRACE_181)

Cột: ['employee', 'emp_salary', 'manager', 'manager_salary', 'qualifies']
   ('Joe', 70000, 'Sam', 60000, 1)
   ('Henry', 80000, 'Max', 90000, 0)
Số dòng: 2


[('Joe', 70000, 'Sam', 60000, 1), ('Henry', 80000, 'Max', 90000, 0)]

## 19. So sánh `INNER JOIN` và `LEFT JOIN` trong bài #181

Nếu dùng **LEFT JOIN** thay INNER JOIN, Sam và Max vẫn xuất hiện, nhưng các cột quản lý (`m.name`, `m.salary`) là NULL.

Nếu sau đó vẫn có `WHERE e.salary > m.salary`, so lương với `NULL` cho kết quả **UNKNOWN**, không thỏa WHERE. Vì vậy **trong chính bài này**, `LEFT JOIN` + `WHERE e.salary > m.salary` thường cho cùng **tập dòng** với INNER JOIN.

Nhưng **INNER JOIN** diễn đạt ý nghĩa đúng và ngắn hơn: ta chỉ cần nhân viên có **quản lý thực sự tìm được**. Đừng suy diễn rằng LEFT JOIN và INNER JOIN luôn tương đương; ở bài #175 chúng khác nhau rõ rệt.

In [15]:
print('LEFT JOIN trước lọc (4 người):')
show_sql_181("""SELECT e.name,m.name,m.salary FROM Employee e
LEFT JOIN Employee m ON e.managerId=m.id ORDER BY e.id;""")
print('LEFT JOIN sau lọc (1 người):')
show_sql_181("""SELECT e.name AS Employee FROM Employee e
LEFT JOIN Employee m ON e.managerId=m.id
WHERE e.salary>m.salary;""")

LEFT JOIN trước lọc (4 người):
Cột: ['name', 'name', 'salary']
   ('Joe', 'Sam', 60000)
   ('Henry', 'Max', 90000)
   ('Sam', None, None)
   ('Max', None, None)
Số dòng: 4
LEFT JOIN sau lọc (1 người):
Cột: ['Employee']
   ('Joe',)
Số dòng: 1


[('Joe',)]

## 20. `NULL` trong SQL: không phải số 0

Khi `managerId = NULL`, ta **không biết/không có** người quản lý trong dữ liệu. SQL không xem NULL là 0.

- `NULL = NULL` không trả True mà trả **UNKNOWN** (hiển thị None trong SQLite).
- `70000 > NULL` cũng là UNKNOWN.
- `WHERE` chỉ giữ dòng khi điều kiện **TRUE**; UNKNOWN bị loại.
- Để kiểm tra NULL phải dùng `IS NULL` hoặc `IS NOT NULL`.

Nhiều bạn nhầm `managerId = NULL` với `managerId = 0` rồi cố tìm nhân viên có `id=0`. **Không được thay NULL thành 0** nếu đề không nói như vậy.

In [16]:
show_sql_181("""SELECT NULL AS missing,
70000 > NULL AS compare_missing,
NULL = NULL AS equal_missing,
NULL IS NULL AS is_missing;""")

Cột: ['missing', 'compare_missing', 'equal_missing', 'is_missing']
   (None, None, None, 1)
Số dòng: 1


[(None, None, None, 1)]

## 21. Nộp SQL đúng định dạng LeetCode

Trên LeetCode #181, chọn **MySQL** (SQL) và dán nguyên câu:

```sql
SELECT e.name AS Employee
FROM Employee AS e
JOIN Employee AS m
    ON e.managerId = m.id
WHERE e.salary > m.salary;
```

**Không có `class Solution`**, không có `import sqlite3`, không có `CREATE TABLE` trong bài submit. `e`, `m` là **bí danh trong một câu SQL**; không bắt buộc đúng chữ đó, nhưng cách đặt tên giúp dễ đọc.

**Không thêm `ORDER BY`** trừ khi đề yêu cầu: đề chấp nhận bất kỳ thứ tự dòng nào.

In [17]:
cursor181=conn181.execute(SQL_SUBMIT_181)
columns181=[x[0] for x in cursor181.description]
rows181=cursor181.fetchall()
assert columns181==['Employee']
assert rows181==[('Joe',)]
print('Đúng một cột Employee, đúng Joe: PASS')

Đúng một cột Employee, đúng Joe: PASS


## 22. Kiểm thử 1: nhân viên có lương BẰNG quản lý

Chữ **“more than”** bắt buộc `>` nghiêm ngặt. Hãy đặt thêm một nhân viên Ana có `salary=60000`, quản lý Sam cũng 60000:

- `60000 > 60000` là sai.
- `60000 >= 60000` là đúng nhưng **không phù hợp đề**.

Ta dùng database bản sao, không thay đổi dữ liệu mẫu chính. Cách dựng **phản ví dụ** này giúp phát hiện lỗi `>=` nhanh hơn hàng trăm ví dụ ngẫu nhiên.

In [18]:
lab_equal=sqlite3.connect(':memory:')
conn181.backup(lab_equal)
lab_equal.execute("INSERT INTO Employee VALUES (5,'Ana',60000,3)")
actual=lab_equal.execute(SQL_SUBMIT_181).fetchall()
assert actual==[('Joe',)]
print('Lương bằng nhau không được chọn:',actual)
lab_equal.close()

Lương bằng nhau không được chọn: [('Joe',)]


## 23. Kiểm thử 2: quản lý cũng là nhân viên của một người khác

Một người có thể vừa **quản lý người khác** vừa **báo cáo cho người trên**. Bài yêu cầu so với **quản lý trực tiếp**, không phải bất kỳ người nào ở cấp cao hơn.

Ví dụ: Leo (lương 110000) báo cáo cho Sam (60000); Sam hiện đang không có người quản lý. Leo phải được chọn. Nếu sau này Sam báo cáo cho Max, ta vẫn so **Leo với Sam**, không nhảy lên Max.

Một lần `JOIN e.managerId=m.id` chính là so với **một cấp quản lý**.

In [19]:
lab_chain=sqlite3.connect(':memory:')
conn181.backup(lab_chain)
lab_chain.execute("UPDATE Employee SET managerId=4 WHERE id=3")
lab_chain.execute("INSERT INTO Employee VALUES (5,'Leo',110000,3)")
rows=set(lab_chain.execute(SQL_SUBMIT_181).fetchall())
assert rows=={('Joe',),('Leo',)}
show_sql_181(SQL_SUBMIT_181,lab_chain)
lab_chain.close()

Cột: ['Employee']
   ('Joe',)
   ('Leo',)
Số dòng: 2


## 24. Kiểm thử 3: tên nhân viên có thể bị trùng!

Trong bảng, **`id` mới là khóa duy nhất**; `name` không được đề bảo đảm duy nhất. Vì thế hai người cùng tên, nếu cả hai thỏa điều kiện, câu SQL có thể trả **hai dòng cùng tên**.

Nếu tự thêm `DISTINCT e.name`, bạn có thể **xóa mất dòng hợp lệ** mà đề không yêu cầu. Đừng thêm `DISTINCT` chỉ để “trông gọn hơn”.

Ta kiểm tra hai người tên Joe: người có id=1 và một người mới tên Joe, cả hai đều có lương lớn hơn quản lý. Truy vấn phải cho **hai dòng Joe**.

In [20]:
lab_same_name=sqlite3.connect(':memory:')
conn181.backup(lab_same_name)
lab_same_name.execute("INSERT INTO Employee VALUES (5,'Joe',95000,4)")
rows=lab_same_name.execute(SQL_SUBMIT_181).fetchall()
assert rows.count(('Joe',))==2
print('Hai người trùng tên đều được giữ:',rows)
lab_same_name.close()

Hai người trùng tên đều được giữ: [('Joe',), ('Joe',)]


## 25. Kiểm thử 4: một nhân viên không có managerId

Nếu `managerId IS NULL`, nhân viên **không thể tham gia cặp self join** theo khóa `managerId=id`. Bởi vậy dù lương cao đến đâu, họ không thể thỏa điều kiện “cao hơn quản lý của mình” khi không có quản lý được xác định.

Sam có lương 60000, Max có lương 90000, nhưng cả hai đều không có quản lý trong mẫu. Thử tăng lương Max lên 1 tỷ: kết quả vẫn không thêm Max. Thuật toán phụ thuộc **quan hệ ghép**, không chỉ giá trị salary.

In [21]:
lab_big_boss=sqlite3.connect(':memory:')
conn181.backup(lab_big_boss)
lab_big_boss.execute('UPDATE Employee SET salary=1000000000 WHERE id=4')
assert lab_big_boss.execute(SQL_SUBMIT_181).fetchall()==[('Joe',)]
print('Max không có quản lý nên không được chọn: PASS')
lab_big_boss.close()

Max không có quản lý nên không được chọn: PASS


## 26. Kiểm thử 5: mã quản lý không tìm thấy dòng tương ứng

Trong dữ liệu chuẩn của hệ thống, `managerId` thường tham chiếu ID tồn tại. Tuy nhiên khi thực hành dữ liệu tự tạo, có thể gặp một `managerId` không xuất hiện trong Employee.

Với **INNER JOIN**, không ghép được cặp nào nên người đó không xuất hiện. Không tự chế mức lương quản lý là 0 để so sánh. Phần test này chỉ giúp hiểu JOIN; **không khẳng định** dữ liệu LeetCode có dòng tham chiếu thiếu.

In [22]:
lab_missing_manager=sqlite3.connect(':memory:')
conn181.backup(lab_missing_manager)
lab_missing_manager.execute("INSERT INTO Employee VALUES (5,'Uma',200000,999)")
assert lab_missing_manager.execute(SQL_SUBMIT_181).fetchall()==[('Joe',)]
print('Không có dòng quản lý khớp: PASS')
lab_missing_manager.close()

Không có dòng quản lý khớp: PASS


## 27. Kiểm thử 6: một bảng rỗng thì sao?

Nếu bảng Employee **rỗng**, không có nhân viên và cũng không có cặp nhân viên–quản lý, truy vấn đúng phải trả **0 dòng**, không phải một dòng NULL.

Đây là bài học phân biệt `SELECT` thông thường và những truy vấn tổng hợp đặc biệt như `COUNT(*)`: với bài này, không có dòng đầu vào thì không có dòng output.

In [23]:
lab_no_emps=sqlite3.connect(':memory:')
conn181.backup(lab_no_emps)
lab_no_emps.execute('DELETE FROM Employee')
assert lab_no_emps.execute(SQL_SUBMIT_181).fetchall()==[]
print('Bảng rỗng: PASS')
lab_no_emps.close()

Bảng rỗng: PASS


## 28. Trace bằng Python cho từng nhân viên — thấy cả lý do bỏ qua

Trong SQL thực tế, query planner có thể thực thi JOIN theo nhiều chiến lược khác nhau. Để học, ta có thể **mô phỏng logic bằng Python**:

- Dùng `managers_by_id` như quyển sổ tra cứu ID → (tên, lương).
- Với từng nhân viên, in `managerId` và thông tin quản lý tìm được.
- Nếu quản lý không tồn tại, ghi rõ “bỏ qua”.
- Nếu tồn tại, in phép so và quyết định lấy / không lấy.

Bảng trace này không phải code nộp SQL; nó giúp người mới có thể **nhìn thấy trạng thái** giống các notebook Two Sum hoặc Roman to Integer.

In [24]:
def trace_salary_compared_to_manager(rows):
    by_id={row[0]:row for row in rows}
    result=[]
    for emp_id,name,salary,manager_id in rows:
        manager=by_id.get(manager_id)
        if manager is None:
            print(name,'-> không có quản lý khớp -> bỏ qua')
            continue
        cond=salary>manager[2]
        print(f'{name}: {salary} > {manager[1]} ({manager[2]}) ? {cond}')
        if cond:
            result.append(name)
    return result

assert trace_salary_compared_to_manager(employee_example)==['Joe']

Joe: 70000 > Sam (60000) ? True
Henry: 80000 > Max (90000) ? False
Sam -> không có quản lý khớp -> bỏ qua
Max -> không có quản lý khớp -> bỏ qua


## 29. Tách vai trò `ON` và `WHERE`: hai câu hỏi khác nhau

Đây là điểm quan trọng nhất của bài:

**`ON e.managerId = m.id`** hỏi: “**Ai là quản lý của nhân viên này?**” và quyết định những **cặp dòng** phù hợp.

**`WHERE e.salary > m.salary`** hỏi: “**Trong cặp đã ghép, nhân viên có lương cao hơn người quản lý không?**” và loại những cặp không đạt.

Nếu hoán đổi tùy tiện các biểu thức, ví dụ `ON e.salary > m.salary` mà không nối theo ID, SQL có thể ghép nhân viên với **nhiều người không phải quản lý**, dẫn đến kết quả sai. `ON` phải thể hiện **quan hệ đúng** trước khi ta so sánh.

In [25]:
WRONG_181="""
SELECT e.name AS Employee,m.name AS supposed_manager
FROM Employee AS e JOIN Employee AS m
ON e.salary > m.salary;
"""
print('Sai: ghép với BẤT KỲ người lương thấp hơn')
show_sql_181(WRONG_181)

Sai: ghép với BẤT KỲ người lương thấp hơn
Cột: ['Employee', 'supposed_manager']
   ('Joe', 'Sam')
   ('Henry', 'Joe')
   ('Henry', 'Sam')
   ('Max', 'Joe')
   ('Max', 'Henry')
   ('Max', 'Sam')
Số dòng: 6


[('Joe', 'Sam'),
 ('Henry', 'Joe'),
 ('Henry', 'Sam'),
 ('Max', 'Joe'),
 ('Max', 'Henry'),
 ('Max', 'Sam')]

## 30. Lỗi hoán nhầm e và m trong phép so lương

Nếu bạn viết:

```sql
WHERE m.salary > e.salary
```
thì đang tìm **nhân viên có quản lý lương cao hơn họ**, tức **ngược hẳn câu hỏi**.

Với mẫu, Henry sẽ xuất hiện (Henry 80000, Max 90000), trong khi Joe biến mất. Đây là một “phản ví dụ” rất mạnh: cùng phép JOIN, chỉ đổi chiều dấu đã đổi nghĩa bài toán.

Mẹo: đọc `e.salary > m.salary` thành **“lương employee lớn hơn lương manager”** trước khi submit.

In [26]:
REVERSED_181="""
SELECT e.name AS Employee
FROM Employee e JOIN Employee m ON e.managerId=m.id
WHERE m.salary > e.salary;
"""
show_sql_181(REVERSED_181)
assert conn181.execute(REVERSED_181).fetchall()==[('Henry',)]

Cột: ['Employee']
   ('Henry',)
Số dòng: 1


## 31. Lỗi đặt tên cột hoặc lấy tên người quản lý

Câu:

```sql
SELECT m.name AS Employee
```

vẫn cho **một cột tên Employee**, nhưng dữ liệu bên trong là **tên quản lý**, không phải tên nhân viên. Ở mẫu sẽ trả `Sam`, sai với đáp án `Joe`.

Cũng có thể viết `SELECT e.name AS name` và lấy đúng dữ liệu Joe, nhưng **sai tên cột output** so với yêu cầu. Phải bảo đảm **đúng nội dung lẫn đúng tên cột**.

In [27]:
BAD_COLUMN_181="""
SELECT m.name AS Employee
FROM Employee e JOIN Employee m ON e.managerId=m.id
WHERE e.salary>m.salary;
"""
show_sql_181(BAD_COLUMN_181)
assert conn181.execute(BAD_COLUMN_181).fetchall()==[('Sam',)]

Cột: ['Employee']
   ('Sam',)
Số dòng: 1


## 32. Biến thể 1: truy vấn cột chênh lệch lương

Trong thực tế, đôi khi người quản lý cần biết **cao hơn bao nhiêu tiền**, không chỉ ai cao hơn. Sau JOIN, ta tính:

```sql
e.salary - m.salary AS salaryGap
```

Với Joe–Sam, `70000 - 60000 = 10000`. Với Henry–Max, `80000 - 90000 = -10000`. Nếu chỉ muốn chênh lệch dương, thêm `WHERE e.salary > m.salary`.

Đây là biến thể để tập **cột biểu thức** và `AS`, không phải output của #181.

In [28]:
show_sql_181("""
SELECT e.name AS employee, m.name AS manager,
       e.salary-m.salary AS salaryGap
FROM Employee e JOIN Employee m ON e.managerId=m.id
ORDER BY e.id;
""")

Cột: ['employee', 'manager', 'salaryGap']
   ('Joe', 'Sam', 10000)
   ('Henry', 'Max', -10000)
Số dòng: 2


[('Joe', 'Sam', 10000), ('Henry', 'Max', -10000)]

## 33. Biến thể 2: liệt kê tên nhân viên và tên quản lý của họ

Đôi khi bài toán không yêu cầu so lương, mà chỉ hỏi cặp “người — quản lý”. Đây là bản **self join thuần**:

```sql
SELECT e.name, m.name
FROM Employee e
JOIN Employee m ON e.managerId = m.id;
```

Điểm quan trọng là **cùng bảng nhưng hai vai trò khác nhau**. Nếu muốn giữ cả người không có quản lý, thay JOIN thành **LEFT JOIN**, và tên quản lý sẽ là NULL. Bài toán xác định có nên giữ họ hay không dựa trên yêu cầu, không phải một nguyên tắc cố định.

In [29]:
print('Chỉ người có quản lý:')
show_sql_181(SQL_MANAGER_PAIRS)
print('Giữ cả người không có quản lý:')
show_sql_181("""SELECT e.name AS employee,m.name AS manager
FROM Employee e LEFT JOIN Employee m ON e.managerId=m.id
ORDER BY e.id;""")

Chỉ người có quản lý:
Cột: ['employee_name', 'manager_name']
   ('Joe', 'Sam')
   ('Henry', 'Max')
Số dòng: 2
Giữ cả người không có quản lý:
Cột: ['employee', 'manager']
   ('Joe', 'Sam')
   ('Henry', 'Max')
   ('Sam', None)
   ('Max', None)
Số dòng: 4


[('Joe', 'Sam'), ('Henry', 'Max'), ('Sam', None), ('Max', None)]

## 34. Biến thể 3: dùng correlated subquery thay JOIN

Một cách khác là truy vấn mức lương quản lý từ **truy vấn con (subquery)**:

```sql
SELECT e.name AS Employee
FROM Employee e
WHERE e.salary > (
    SELECT m.salary FROM Employee m WHERE m.id=e.managerId
);
```

Nó diễn đạt đúng: “lương e lớn hơn lương của nhân viên có id bằng managerId của e”. Đây là **correlated subquery** (truy vấn con tham chiếu tới dòng ngoài, ở đây là `e.managerId`).

Cách này cũng đúng trong bài này với `id` là khóa chính; **không mặc định nó nhanh hơn** JOIN. Database có thể tối ưu lại plan; độ phức tạp phụ thuộc chỉ mục và bộ máy. Với người mới, lời giải `JOIN` dễ trace hơn nên vẫn là lời giải chính.

In [30]:
SQL_SUBQUERY_181="""
SELECT e.name AS Employee
FROM Employee e
WHERE e.salary > (
    SELECT m.salary FROM Employee m WHERE m.id=e.managerId
);
"""
assert conn181.execute(SQL_SUBQUERY_181).fetchall()==[('Joe',)]
show_sql_181(SQL_SUBQUERY_181)

Cột: ['Employee']
   ('Joe',)
Số dòng: 1


[('Joe',)]

## 35. Vì sao thuật toán đúng? Chứng minh bằng từng dòng

Ta cần chỉ ra rằng truy vấn **không bỏ sót và không lấy nhầm**:

1. Với một nhân viên e có quản lý m trong bảng, điều kiện `e.managerId=m.id` xác định **đúng dòng quản lý trực tiếp**; do `id` là khóa chính nên m được xác định duy nhất.
2. `INNER JOIN` tạo cặp `(e,m)` nếu tìm thấy m. Người không có m không tạo cặp; họ không thể thỏa phép so lương với quản lý.
3. `WHERE e.salary > m.salary` **chỉ giữ** những cặp mà lương nhân viên **lớn hơn nghiêm ngặt** lương quản lý.
4. `SELECT e.name AS Employee` xuất **tên đúng người thỏa điều kiện**, với tên cột chính xác.

Do đó mọi nhân viên thỏa đề sẽ xuất hiện, mọi người không thỏa đều bị loại — truy vấn là đúng.

## 36. Big-O với SQL self join: phụ thuộc kế hoạch thực thi

Đặt `n` là số dòng Employee và `k` là số dòng output.

- **Python baseline** dùng hai vòng lặp: O(n²) thời gian trong tình huống tổng quát, O(k) bộ nhớ để lưu kết quả.
- **Phòng lab trace** dùng dictionary ID→row: tạo từ điển O(n) bộ nhớ, tra ID O(1) trung bình; quét nhân viên O(n) thời gian kỳ vọng (không tính số dòng kết quả).
- **SQL JOIN** không có một Big-O cố định: có chỉ mục phù hợp trên `Employee.id` (khóa chính) thì database có thể tra cứu quản lý hiệu quả; hash join có thể gần O(n+k) thời gian kỳ vọng và dùng O(n) bộ nhớ; nested-loop không tối ưu có thể O(n²).
- `SELECT` phải xuất k kết quả, ít nhất tốn O(k) chi phí xuất.

**Không viết “self join chắc chắn O(n) / O(1) bộ nhớ”** như thể nó là thuật toán Python thông thường. Hãy nói rõ giả định về engine/index khi phân tích SQL.

## 37. Những lỗi điển hình, phản ví dụ và cách sửa

| Sai lầm | Vì sao sai | Cách sửa |
|---|---|---|
| `JOIN Manager m` | Đề không có bảng Manager | `JOIN Employee m` |
| `ON e.id = m.managerId` | Ghép **quản lý với cấp dưới** ở chiều ngược | `ON e.managerId = m.id` |
| `ON e.salary > m.salary` | Ghép với người lương thấp bất kỳ, không phải quản lý | JOIN theo ID, so lương ở WHERE |
| `WHERE m.salary > e.salary` | Lọc người có quản lý giàu hơn | `e.salary > m.salary` |
| `WHERE e.salary >= m.salary` | Giữ cả lương bằng nhau | Dùng `>` |
| `SELECT m.name AS Employee` | Xuất tên quản lý | `SELECT e.name AS Employee` |
| `SELECT e.name AS name` | Sai tên cột bắt buộc | `AS Employee` |
| `SELECT DISTINCT e.name` | Có thể xóa các dòng hợp lệ của người trùng tên | Không thêm DISTINCT khi đề không yêu cầu |

Hãy nhớ hai vai trò riêng biệt: **ON tìm đúng người**, **WHERE kiểm tra đúng quan hệ lương**.

# 38. Bài tập tự luyện — 6 bài từ dễ tới khó

**Không xem lời giải bên dưới ngay.** Bài SQL cần **tự tính trên dữ liệu**, sau đó **tự viết truy vấn**, rồi mới chạy test. Các cell TODO chỉ tạo chuỗi comment, không tự gọi SQL.

1. **Dự đoán output:** `SELECT name FROM Employee WHERE salary > 70000` có ai? Vì sao Joe không có trong danh sách?
2. **Trace bằng tay:** Lập cặp `(employee, manager, employee_salary, manager_salary)` cho Joe và Henry. Dòng nào đạt `>`?
3. **Điền câu SQL:** hoàn thiện `SELECT ... FROM Employee e JOIN Employee m ON ... WHERE ...`.
4. **Viết từ đầu:** truy vấn liệt kê **tên nhân viên và tên quản lý** của người có quản lý (không so lương).
5. **Biến thể:** tìm nhân viên có lương **thấp hơn** quản lý, trả về cột `Employee`.
6. **Mini challenge:** tìm nhân viên có lương **cao hơn quản lý ít nhất 5000**, xuất `Employee` và `salaryGap`, sắp xếp theo ID nhân viên để tự kiểm dễ dàng.

**Test gợi ý** được comment ở dưới từng bài; chỉ bỏ dấu `#` sau khi viết xong SQL.

In [31]:
# Bài 1,2: tính bằng tay trước. Không chạy đáp án tự động.
practice_employees=['Joe','Henry','Sam','Max']

### Bài 3. TODO: ghép đúng người và so đúng lương

Phải phân biệt **một điều kiện ghép** (ID) và **một điều kiện lọc** (salary). Tên cột kết quả là `Employee`, không phải `name`.

In [32]:
EX_181_3="""
-- TODO: SELECT e.name AS Employee
-- TODO: FROM Employee e JOIN Employee m
-- TODO: ON e.managerId=m.id
-- TODO: WHERE e.salary>m.salary
"""
# Khi làm xong mới bật:
# assert conn181.execute(EX_181_3).fetchall() == [('Joe',)]

### Bài 4. Tự viết danh sách nhân viên và quản lý

Kết quả mong muốn: hai cột `employee, manager`; hai dòng Joe–Sam, Henry–Max. Chỉ cần `JOIN ON` theo ID, **không cần WHERE so lương**.

In [33]:
EX_181_4="""
-- TODO: SELECT e.name AS employee,m.name AS manager
-- TODO: FROM Employee e JOIN Employee m ON ...
"""
# assert set(conn181.execute(EX_181_4).fetchall()) == {('Joe','Sam'),('Henry','Max')}

### Bài 5. Lọc người lương thấp hơn quản lý

Nếu bạn hiểu rõ tên biến, chỉ cần đảo **chiều dấu so sánh**, không cần thay điều kiện ON. Kết quả mẫu phải là **Henry**.

In [34]:
EX_181_5="""
-- TODO: JOIN đúng như cũ; thay WHERE để lấy salary nhân viên < manager
"""
# assert conn181.execute(EX_181_5).fetchall() == [('Henry',)]

### Bài 6. Mini challenge: lương cao hơn quản lý ít nhất 5000

Ta không còn hỏi `e.salary > m.salary` mà hỏi **chênh lệch**:

```text
salaryGap = e.salary - m.salary
```

Điều kiện `salaryGap >= 5000`, kết quả có hai cột `Employee` và `salaryGap`. Với mẫu, Joe được chọn với chênh lệch 10000. Thêm `ORDER BY e.id` chỉ để test dễ nhìn.

In [35]:
EX_181_6="""
-- TODO: SELECT e.name AS Employee,(e.salary-m.salary) AS salaryGap
-- TODO: INNER JOIN theo managerId; WHERE chênh lệch >= 5000
-- TODO: ORDER BY e.id
"""
# assert conn181.execute(EX_181_6).fetchall() == [('Joe',10000)]

## 39. Gợi ý 3 mức cho từng bài

| Bài | Mức 1 — nhắc ý | Mức 2 — nhắc cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | So với 70000 cố định | `WHERE salary > 70000` | Henry, Max; Joe đúng bằng 70000 |
| 2 | Manager ID dẫn tới dòng nào? | Joe→3, Henry→4 | Joe–Sam đúng; Henry–Max sai |
| 3 | Ghép rồi lọc | `JOIN Employee m ON e.managerId=m.id` | `WHERE e.salary>m.salary` |
| 4 | Chỉ cần cặp đúng | self join + `SELECT` 2 tên | `SELECT e.name AS employee,m.name AS manager` |
| 5 | Đổi chiều so sánh | `e.salary < m.salary` | Kết quả Henry |
| 6 | So phần chênh lệch | `(e.salary-m.salary)>=5000` | `SELECT e.name AS Employee, e.salary-m.salary AS salaryGap` |

**Chỉ chuyển đến đáp án khi đã thử tự viết.**

## 40. Đáp án Bài 1–2 (đọc sau khi tính tay)

**Bài 1:** Henry (80000) và Max (90000). Joe **không** xuất hiện vì điều kiện là `salary > 70000`, trong khi Joe có lương **bằng** 70000.

**Bài 2:**

- Joe–Sam: `(Joe, Sam, 70000, 60000)`, phép so **True**.
- Henry–Max: `(Henry, Max, 80000, 90000)`, phép so **False**.

Một mẹo học SQL: trước khi viết truy vấn cuối, thử liệt kê **bảng trung gian** — lúc này lỗi ghép sai ID sẽ lộ ra sớm.

In [36]:
assert conn181.execute('SELECT name FROM Employee WHERE salary>70000 ORDER BY id').fetchall()==[('Henry',),('Max',)]
assert conn181.execute(SQL_MANAGER_PAIRS).fetchall()==[('Joe','Sam'),('Henry','Max')]
print('Bài 1 và 2: PASS')

Bài 1 và 2: PASS


## 41. Đáp án Bài 3 — truy vấn chuẩn bài #181

Mỗi dòng có vai trò rõ ràng: `e.name` xuất nhân viên; `JOIN` tìm quản lý; `WHERE` kiểm tra lương. Không cần thêm DISTINCT hoặc ORDER BY.

In [37]:
ANSWER_181_3="""
SELECT e.name AS Employee
FROM Employee AS e
JOIN Employee AS m ON e.managerId = m.id
WHERE e.salary > m.salary;
"""
assert conn181.execute(ANSWER_181_3).fetchall()==[('Joe',)]
show_sql_181(ANSWER_181_3)

Cột: ['Employee']
   ('Joe',)
Số dòng: 1


[('Joe',)]

## 42. Đáp án Bài 4 — xuất tên hai vai trò

Vẫn là cùng một bảng Employee, nhưng nhờ **bí danh e/m**, ta đọc chính xác nhân viên và quản lý. Không cần `WHERE` vì bài chỉ hỏi “ai là quản lý của ai”.

In [38]:
ANSWER_181_4="""
SELECT e.name AS employee,m.name AS manager
FROM Employee e
JOIN Employee m ON e.managerId=m.id;
"""
assert set(conn181.execute(ANSWER_181_4).fetchall())=={('Joe','Sam'),('Henry','Max')}
show_sql_181(ANSWER_181_4)

Cột: ['employee', 'manager']
   ('Joe', 'Sam')
   ('Henry', 'Max')
Số dòng: 2


[('Joe', 'Sam'), ('Henry', 'Max')]

## 43. Đáp án Bài 5 — thay đổi chiều so lương

Chỉ khác `WHERE e.salary < m.salary`; ON **không đổi** vì quan hệ nhân viên và quản lý không đổi. Đây là một ví dụ phân biệt điều kiện **quan hệ** và điều kiện **lọc**.

In [39]:
ANSWER_181_5="""
SELECT e.name AS Employee
FROM Employee e
JOIN Employee m ON e.managerId=m.id
WHERE e.salary < m.salary;
"""
assert conn181.execute(ANSWER_181_5).fetchall()==[('Henry',)]
show_sql_181(ANSWER_181_5)

Cột: ['Employee']
   ('Henry',)
Số dòng: 1


[('Henry',)]

## 44. Đáp án Bài 6 — hiệu lương và ngưỡng 5000

Biểu thức `e.salary - m.salary` là một giá trị SQL có thể dùng **cả ở SELECT lẫn WHERE**. Không bắt buộc database cho phép sử dụng bí danh `salaryGap` trực tiếp trong WHERE; để dễ tương thích, ta **viết lại biểu thức đầy đủ** trong WHERE.

In [40]:
ANSWER_181_6="""
SELECT e.name AS Employee, e.salary-m.salary AS salaryGap
FROM Employee e
JOIN Employee m ON e.managerId=m.id
WHERE e.salary-m.salary>=5000
ORDER BY e.id;
"""
assert conn181.execute(ANSWER_181_6).fetchall()==[('Joe',10000)]
show_sql_181(ANSWER_181_6)

Cột: ['Employee', 'salaryGap']
   ('Joe', 10000)
Số dòng: 1


[('Joe', 10000)]

## 45. Tóm tắt tự học và checklist submit

**Template nộp LeetCode, chỉ dán đoạn SQL dưới:**
```sql
SELECT e.name AS Employee
FROM Employee AS e
JOIN Employee AS m
    ON e.managerId = m.id
WHERE e.salary > m.salary;
```

**Pattern nhận diện:** “Mỗi dòng trong bảng có một cột ID trỏ đến **dòng khác cùng bảng**” → nghĩ đến **self join**. Ví dụ: nhân viên–quản lý, danh mục–danh mục cha, thành viên–người giới thiệu.

**Checklist:**
- [ ] Mình hiểu `managerId` trỏ tới **`Employee.id`**, không phải `Employee.managerId`?
- [ ] Có **hai alias** đại diện cho hai vai trò (`e` và `m`)?
- [ ] `ON` dùng để ghép đúng nhân viên–quản lý, `WHERE` dùng để so lương?
- [ ] Lương **bằng** nhau phải bị loại vì đề yêu cầu **cao hơn**?
- [ ] `SELECT e.name AS Employee` xuất tên **nhân viên**, đúng tên cột?
- [ ] Không thêm DISTINCT, ORDER BY hoặc cột phụ khi đề không yêu cầu?
- [ ] Đã kiểm thử Joe, Henry, managerId=NULL, salary bằng nhau, trùng tên?
- [ ] Chỉ nộp **SQL**, không dán `sqlite3` từ notebook?

**Một câu cần nhớ:** “Cùng bảng nhưng hai vai trò: dùng alias để tìm quản lý trực tiếp bằng ID, rồi so lương e với lương m.”